# 6교시. OCR 및 정보 추출 기능 연동

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/06_ocr_ai_integration.ipynb)

**이번 교시 행동:** 업로드한 파일을 실제 OCR 함수에 연결하고 LIVE·오류·복구 모드를 화면에서 구분합니다.

**통과 증거:** `course_outputs/app_06.py`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )


In [ ]:
from textwrap import dedent

app_code = 'import tempfile\nfrom pathlib import Path\nimport streamlit as st\n\nGOLDEN_OCR_TEXT = \'이태리집\\n거래일시 2025-10-04 12:33:37\\n페퍼로니 앤 치즈 29,000 1 29,000\\n토마토 파스타 14,000 1 14,000\\n수제 돈가스 13,000 1 13,000\\n새우 칠리치 필라 14,000 1 14,000\\n콜라 2,000 3 6,000\\n합계 금액 76,000\\n부가세 과세물품가액 69,094\\n부가세 6,906\\n\'\nGOLDEN_RECEIPT = {\'document_type\': \'receipt\', \'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000, \'items\': [{\'name\': \'페퍼로니 앤 치즈\', \'quantity\': 1, \'unit_price\': 29000, \'line_total\': 29000}, {\'name\': \'토마토 파스타\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'수제 돈가스\', \'quantity\': 1, \'unit_price\': 13000, \'line_total\': 13000}, {\'name\': \'새우 칠리치 필라\', \'quantity\': 1, \'unit_price\': 14000, \'line_total\': 14000}, {\'name\': \'콜라\', \'quantity\': 3, \'unit_price\': 2000, \'line_total\': 6000}], \'adjustments\': {\'discount\': 0, \'tax\': 0, \'service\': 0, \'rounding\': 0}, \'tax_breakdown\': {\'mode\': \'included_in_item_prices\', \'supply_amount\': 69094, \'vat\': 6906, \'payable_total\': 76000}, \'raw_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04 12:33:37\', \'total_amount\': \'76,000\'}, \'cleaned_values\': {\'store_name\': \'이태리집\', \'date\': \'2025-10-04\', \'total_amount\': 76000}, \'evidence\': {\'store_name\': {\'raw_value\': \'이태리집\', \'line\': 1}, \'date\': {\'raw_value\': \'거래일시 2025-10-04 12:33:37\', \'line\': 2}, \'total_amount\': {\'raw_value\': \'합계 금액 76,000\', \'line\': 8}}, \'source_mode\': \'prepared_fixture_rule_extraction\'}\n\nfrom collections import defaultdict\n\ndef reconstruct_spatial_lines(items):\n    positioned_by_page = defaultdict(list)\n    unpositioned_by_page = defaultdict(list)\n    for order, item in enumerate(items):\n        text = " ".join(str(item.get("text", "")).split())\n        if not text:\n            continue\n        page = int(item.get("page") or 1)\n        points = [\n            point\n            for point in (item.get("box") or [])\n            if isinstance(point, (list, tuple)) and len(point) >= 2\n        ]\n        if not points:\n            unpositioned_by_page[page].append((order, text))\n            continue\n        xs = [float(point[0]) for point in points]\n        ys = [float(point[1]) for point in points]\n        positioned_by_page[page].append({\n            "text": text,\n            "x": min(xs),\n            "y": sum(ys) / len(ys),\n            "height": max(ys) - min(ys),\n            "order": order,\n        })\n\n    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))\n    lines = []\n    for page in pages:\n        rows = []\n        for token in sorted(\n            positioned_by_page[page],\n            key=lambda value: (value["y"], value["x"], value["order"]),\n        ):\n            row = rows[-1] if rows else None\n            tolerance = (\n                max(12.0, min(24.0, max(row["height"], token["height"]) * 0.45))\n                if row else 12.0\n            )\n            if row and abs(token["y"] - row["y"]) <= tolerance:\n                row["tokens"].append(token)\n                count = len(row["tokens"])\n                row["y"] = (row["y"] * (count - 1) + token["y"]) / count\n                row["height"] = max(row["height"], token["height"])\n            else:\n                rows.append({\n                    "tokens": [token],\n                    "y": token["y"],\n                    "height": token["height"],\n                })\n\n        lines.extend(\n            " ".join(\n                token["text"]\n                for token in sorted(\n                    row["tokens"],\n                    key=lambda value: (value["x"], value["order"]),\n                )\n            )\n            for row in rows\n        )\n        lines.extend(\n            text\n            for _, text in sorted(\n                unpositioned_by_page[page],\n                key=lambda value: value[0],\n            )\n        )\n    return lines\n\nimport re\n\ndef to_int(value):\n    return int(value.replace(",", ""))\n\n\ndef extract_receipt_from_text(text, source_mode):\n    lines = [line.strip() for line in text.splitlines() if line.strip()]\n    date_match = re.search(r"\\b(\\d{4})[-./](\\d{1,2})[-./](\\d{1,2})\\b", text)\n    total_line = next(\n        (\n            line\n            for line in lines\n            if re.search(r"(?:합\\s*계|결제\\s*금액|총\\s*액)", line)\n        ),\n        None,\n    )\n    total_candidates = (\n        re.findall(r"(?<![\\d,])\\d[\\d,]*(?![\\d,])", total_line)\n        if total_line\n        else []\n    )\n    total_raw = total_candidates[-1] if total_candidates else None\n    supply_match = re.search(\n        r"(?:부가세\\s*)?과세물품가액\\s*[:：]?\\s*([\\d,]+)",\n        text,\n    )\n    vat_match = re.search(\n        r"^부가세(?!\\s*과세물품가액)\\s*[:：]?\\s*([\\d,]+)",\n        text,\n        re.MULTILINE,\n    )\n    item_pattern = re.compile(\n        r"^(?P<name>.+?)\\s+(?P<unit>[\\d,]+)\\s+"\n        r"(?P<quantity>\\d+)\\s+(?P<line>[\\d,]+)$"\n    )\n    markdown_item_pattern = re.compile(\n        r"^\\|\\s*(?P<name>[^|]+?)\\s*\\|\\s*(?P<quantity>\\d+)\\s*\\|"\n        r"\\s*(?P<unit>[\\d,]+)원\\s*\\|\\s*(?P<line>[\\d,]+)원\\s*\\|$"\n    )\n    items = []\n    item_evidence = []\n    for line_number, line in enumerate(lines, start=1):\n        match = item_pattern.search(line)\n        if not match:\n            match = markdown_item_pattern.search(line)\n        if match:\n            item = {\n                "name": match.group("name"),\n                "quantity": int(match.group("quantity")),\n                "unit_price": to_int(match.group("unit")),\n                "line_total": to_int(match.group("line")),\n            }\n            items.append(item)\n            item_evidence.append({"line": line_number, "raw_value": line})\n\n    date_value = (\n        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"\n        f"{int(date_match.group(3)):02d}"\n        if date_match else None\n    )\n    total_value = to_int(total_raw) if total_raw else None\n    supply_value = to_int(supply_match.group(1)) if supply_match else None\n    vat_value = to_int(vat_match.group(1)) if vat_match else None\n    return {\n        "document_type": "receipt",\n        "store_name": lines[0] if lines else None,\n        "date": date_value,\n        "total_amount": total_value,\n        "items": items,\n        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},\n        "tax_breakdown": {\n            "mode": "included_in_item_prices",\n            "supply_amount": supply_value,\n            "vat": vat_value,\n            "payable_total": total_value,\n        } if supply_value is not None and vat_value is not None else None,\n        "raw_values": {\n            "store_name": lines[0] if lines else None,\n            "date": date_match.group(0) if date_match else None,\n            "total_amount": total_raw,\n        },\n        "cleaned_values": {\n            "store_name": lines[0] if lines else None,\n            "date": date_value,\n            "total_amount": total_value,\n        },\n        "evidence": {\n            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},\n            "date": {"raw_value": date_match.group(0) if date_match else None},\n            "total_amount": {"raw_value": total_line},\n            "items": item_evidence,\n        },\n        "source_mode": source_mode,\n    }\n\ndef run_live_ocr(uploaded):\n    suffix = Path(uploaded.name).suffix.lower()\n    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as temp:\n        temp.write(uploaded.getvalue())\n        path = temp.name\n    try:\n        from paddleocr import PaddleOCR\n        engine = PaddleOCR(\n            lang="korean",\n            ocr_version="PP-OCRv5",\n            use_doc_orientation_classify=False,\n            use_doc_unwarping=False,\n            use_textline_orientation=False,\n            device="cpu",\n        )\n        page = list(engine.predict(path))[0]\n        payload = page.json() if callable(page.json) else page.json\n        result = payload.get("res", payload)\n        items = [\n            {\n                "page": 1,\n                "box": box.tolist() if hasattr(box, "tolist") else box,\n                "text": text,\n                "confidence": float(score),\n            }\n            for box, text, score in zip(\n                result.get("rec_polys", []),\n                result.get("rec_texts", []),\n                result.get("rec_scores", []),\n            )\n        ]\n        return "\\n".join(reconstruct_spatial_lines(items))\n    finally:\n        Path(path).unlink(missing_ok=True)\n\n\ndef process_document(uploaded=None, *, use_prepared=False):\n    if use_prepared:\n        text = GOLDEN_OCR_TEXT\n        mode = "PREPARED_FALLBACK"\n    elif uploaded is None:\n        return {"ok": False, "mode": "INPUT_ERROR", "error": "파일을 선택하세요."}\n    else:\n        try:\n            text = run_live_ocr(uploaded)\n            mode = "LIVE"\n        except Exception as exc:\n            return {\n                "ok": False,\n                "mode": "LIVE_ERROR",\n                "error": f"{type(exc).__name__}: {exc}",\n                "recovery": "공개 샘플 준비 결과 버튼을 선택하세요.",\n            }\n    data = extract_receipt_from_text(\n        text,\n        "live_ocr_rule_extraction" if mode == "LIVE"\n        else "prepared_fixture_rule_extraction",\n    )\n    return {"ok": True, "mode": mode, "ocr_text": text, "data": data}\n\n\nst.title("영수증 Document AI 연결 앱")\nuploaded = st.file_uploader(\n    "승인된 비식별 이미지 또는 PDF 한 장 · 최대 5MB",\n    type=["png", "jpg", "jpeg", "pdf"],\n    max_upload_size=5,\n    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n)\nleft, right = st.columns(2)\nrun_live = left.button("업로드 파일 LIVE 처리")\nrun_prepared = right.button("공개 샘플 준비 결과")\nif run_live or run_prepared:\n    result = process_document(uploaded, use_prepared=run_prepared)\n    if result["ok"]:\n        st.success(f"실행 모드: {result[\'mode\']}")\n        st.text_area("OCR 원문", result["ocr_text"], height=220)\n        st.json(result["data"])\n    else:\n        st.error(f"{result[\'mode\']} · {result[\'error\']}")\n        if result.get("recovery"):\n            st.info(result["recovery"])\n'
output_path = OUTPUT_DIR / "app_06.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장:", output_path)


In [ ]:
from collections import defaultdict

def reconstruct_spatial_lines(items):
    positioned_by_page = defaultdict(list)
    unpositioned_by_page = defaultdict(list)
    for order, item in enumerate(items):
        text = " ".join(str(item.get("text", "")).split())
        if not text:
            continue
        page = int(item.get("page") or 1)
        points = [
            point
            for point in (item.get("box") or [])
            if isinstance(point, (list, tuple)) and len(point) >= 2
        ]
        if not points:
            unpositioned_by_page[page].append((order, text))
            continue
        xs = [float(point[0]) for point in points]
        ys = [float(point[1]) for point in points]
        positioned_by_page[page].append({
            "text": text,
            "x": min(xs),
            "y": sum(ys) / len(ys),
            "height": max(ys) - min(ys),
            "order": order,
        })

    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))
    lines = []
    for page in pages:
        rows = []
        for token in sorted(
            positioned_by_page[page],
            key=lambda value: (value["y"], value["x"], value["order"]),
        ):
            row = rows[-1] if rows else None
            tolerance = (
                max(12.0, min(24.0, max(row["height"], token["height"]) * 0.45))
                if row else 12.0
            )
            if row and abs(token["y"] - row["y"]) <= tolerance:
                row["tokens"].append(token)
                count = len(row["tokens"])
                row["y"] = (row["y"] * (count - 1) + token["y"]) / count
                row["height"] = max(row["height"], token["height"])
            else:
                rows.append({
                    "tokens": [token],
                    "y": token["y"],
                    "height": token["height"],
                })

        lines.extend(
            " ".join(
                token["text"]
                for token in sorted(
                    row["tokens"],
                    key=lambda value: (value["x"], value["order"]),
                )
            )
            for row in rows
        )
        lines.extend(
            text
            for _, text in sorted(
                unpositioned_by_page[page],
                key=lambda value: value[0],
            )
        )
    return lines


import re

def to_int(value):
    return int(value.replace(",", ""))


def extract_receipt_from_text(text, source_mode):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    date_match = re.search(r"\b(\d{4})[-./](\d{1,2})[-./](\d{1,2})\b", text)
    total_line = next(
        (
            line
            for line in lines
            if re.search(r"(?:합\s*계|결제\s*금액|총\s*액)", line)
        ),
        None,
    )
    total_candidates = (
        re.findall(r"(?<![\d,])\d[\d,]*(?![\d,])", total_line)
        if total_line
        else []
    )
    total_raw = total_candidates[-1] if total_candidates else None
    supply_match = re.search(
        r"(?:부가세\s*)?과세물품가액\s*[:：]?\s*([\d,]+)",
        text,
    )
    vat_match = re.search(
        r"^부가세(?!\s*과세물품가액)\s*[:：]?\s*([\d,]+)",
        text,
        re.MULTILINE,
    )
    item_pattern = re.compile(
        r"^(?P<name>.+?)\s+(?P<unit>[\d,]+)\s+"
        r"(?P<quantity>\d+)\s+(?P<line>[\d,]+)$"
    )
    markdown_item_pattern = re.compile(
        r"^\|\s*(?P<name>[^|]+?)\s*\|\s*(?P<quantity>\d+)\s*\|"
        r"\s*(?P<unit>[\d,]+)원\s*\|\s*(?P<line>[\d,]+)원\s*\|$"
    )
    items = []
    item_evidence = []
    for line_number, line in enumerate(lines, start=1):
        match = item_pattern.search(line)
        if not match:
            match = markdown_item_pattern.search(line)
        if match:
            item = {
                "name": match.group("name"),
                "quantity": int(match.group("quantity")),
                "unit_price": to_int(match.group("unit")),
                "line_total": to_int(match.group("line")),
            }
            items.append(item)
            item_evidence.append({"line": line_number, "raw_value": line})

    date_value = (
        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"
        f"{int(date_match.group(3)):02d}"
        if date_match else None
    )
    total_value = to_int(total_raw) if total_raw else None
    supply_value = to_int(supply_match.group(1)) if supply_match else None
    vat_value = to_int(vat_match.group(1)) if vat_match else None
    return {
        "document_type": "receipt",
        "store_name": lines[0] if lines else None,
        "date": date_value,
        "total_amount": total_value,
        "items": items,
        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},
        "tax_breakdown": {
            "mode": "included_in_item_prices",
            "supply_amount": supply_value,
            "vat": vat_value,
            "payable_total": total_value,
        } if supply_value is not None and vat_value is not None else None,
        "raw_values": {
            "store_name": lines[0] if lines else None,
            "date": date_match.group(0) if date_match else None,
            "total_amount": total_raw,
        },
        "cleaned_values": {
            "store_name": lines[0] if lines else None,
            "date": date_value,
            "total_amount": total_value,
        },
        "evidence": {
            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},
            "date": {"raw_value": date_match.group(0) if date_match else None},
            "total_amount": {"raw_value": total_line},
            "items": item_evidence,
        },
        "source_mode": source_mode,
    }


RECORDED_PP_OCRV5_TOKENS = [{'box': [[306, 98], [438, 98], [438, 141], [306, 141]], 'text': '[영수', 'confidence': 0.86885666847229, 'matches_source': None, 'review_note': ''}, {'box': [[427, 102], [487, 102], [487, 138], [427, 138]], 'text': '', 'confidence': 0.0, 'matches_source': None, 'review_note': ''}, {'box': [[70, 174], [261, 179], [259, 218], [69, 212]], 'text': '이태리쉽!', 'confidence': 0.8561691045761108, 'matches_source': None, 'review_note': ''}, {'box': [[67, 213], [833, 218], [832, 267], [66, 262]], 'text': '강원특별자지도 태시민영로 262(장시동)', 'confidence': 0.7325795888900757, 'matches_source': None, 'review_note': ''}, {'box': [[66, 342], [421, 347], [420, 384], [65, 379]], 'text': '2025-10-04 12:33:37', 'confidence': 0.9963311553001404, 'matches_source': None, 'review_note': ''}, {'box': [[143, 423], [337, 423], [337, 475], [143, 475]], 'text': '상  명', 'confidence': 0.7398265600204468, 'matches_source': None, 'review_note': ''}, {'box': [[456, 431], [498, 431], [498, 471], [456, 471]], 'text': '11', 'confidence': 0.5209168791770935, 'matches_source': None, 'review_note': ''}, {'box': [[513, 431], [550, 431], [550, 471], [513, 471]], 'text': '1|', 'confidence': 0.7733569145202637, 'matches_source': None, 'review_note': ''}, {'box': [[586, 429], [664, 429], [664, 475], [586, 475]], 'text': '수5', 'confidence': 0.3517782688140869, 'matches_source': None, 'review_note': ''}, {'box': [[753, 427], [799, 427], [799, 474], [753, 474]], 'text': '금', 'confidence': 0.5398801565170288, 'matches_source': None, 'review_note': ''}, {'box': [[808, 428], [852, 428], [852, 469], [808, 469]], 'text': '액', 'confidence': 0.6926582455635071, 'matches_source': None, 'review_note': ''}, {'box': [[70, 507], [367, 510], [366, 551], [69, 549]], 'text': '페퍼르니인집스', 'confidence': 0.6701959371566772, 'matches_source': None, 'review_note': ''}, {'box': [[436, 510], [552, 510], [552, 549], [436, 549]], 'text': '29,000', 'confidence': 0.9894272685050964, 'matches_source': None, 'review_note': ''}, {'box': [[623, 513], [643, 513], [643, 546], [623, 546]], 'text': '1', 'confidence': 0.9818213582038879, 'matches_source': None, 'review_note': ''}, {'box': [[732, 511], [854, 511], [854, 551], [732, 551]], 'text': '29,000', 'confidence': 0.9425509572029114, 'matches_source': None, 'review_note': ''}, {'box': [[73, 550], [314, 552], [313, 592], [72, 589]], 'text': '토마ㅌ마스', 'confidence': 0.7026697397232056, 'matches_source': None, 'review_note': ''}, {'box': [[439, 551], [552, 551], [552, 590], [439, 590]], 'text': '14000', 'confidence': 0.9957329630851746, 'matches_source': None, 'review_note': ''}, {'box': [[621, 553], [644, 553], [644, 588], [621, 588]], 'text': '1', 'confidence': 0.913772702217102, 'matches_source': None, 'review_note': ''}, {'box': [[735, 553], [854, 553], [854, 594], [735, 594]], 'text': '14,000', 'confidence': 0.9957665801048279, 'matches_source': None, 'review_note': ''}, {'box': [[71, 588], [321, 588], [321, 634], [71, 634]], 'text': '수제돈가스', 'confidence': 0.9752224087715149, 'matches_source': None, 'review_note': ''}, {'box': [[440, 591], [552, 591], [552, 631], [440, 631]], 'text': '13,000', 'confidence': 0.9718451499938965, 'matches_source': None, 'review_note': ''}, {'box': [[621, 595], [645, 595], [645, 629], [621, 629]], 'text': '1', 'confidence': 0.999768078327179, 'matches_source': None, 'review_note': ''}, {'box': [[736, 592], [854, 595], [853, 635], [735, 633]], 'text': '13,000', 'confidence': 0.9974907040596008, 'matches_source': None, 'review_note': ''}, {'box': [[72, 635], [151, 635], [151, 672], [72, 672]], 'text': '새우', 'confidence': 0.9985453486442566, 'matches_source': None, 'review_note': ''}, {'box': [[162, 632], [368, 632], [368, 673], [162, 673]], 'text': '실시치리', 'confidence': 0.8055554032325745, 'matches_source': None, 'review_note': ''}, {'box': [[440, 632], [552, 632], [552, 671], [440, 671]], 'text': '14,000', 'confidence': 0.9835992455482483, 'matches_source': None, 'review_note': ''}, {'box': [[623, 637], [643, 637], [643, 669], [623, 669]], 'text': '1', 'confidence': 0.9788281321525574, 'matches_source': None, 'review_note': ''}, {'box': [[738, 636], [853, 636], [853, 676], [738, 676]], 'text': '14,000', 'confidence': 0.987253725528717, 'matches_source': None, 'review_note': ''}, {'box': [[71, 673], [150, 673], [150, 716], [71, 716]], 'text': '콜라', 'confidence': 0.9953886270523071, 'matches_source': None, 'review_note': ''}, {'box': [[455, 674], [552, 674], [552, 712], [455, 712]], 'text': '2,000', 'confidence': 0.9925152659416199, 'matches_source': None, 'review_note': ''}, {'box': [[620, 676], [644, 676], [644, 713], [620, 713]], 'text': '3', 'confidence': 0.9491386413574219, 'matches_source': None, 'review_note': ''}, {'box': [[748, 674], [855, 677], [854, 721], [747, 719]], 'text': '6,000', 'confidence': 0.995599627494812, 'matches_source': None, 'review_note': ''}, {'box': [[70, 753], [170, 753], [170, 799], [70, 799]], 'text': '합계', 'confidence': 0.9771654605865479, 'matches_source': None, 'review_note': ''}, {'box': [[200, 751], [300, 751], [300, 801], [200, 801]], 'text': '9', 'confidence': 0.1812758445739746, 'matches_source': None, 'review_note': ''}, {'box': [[733, 756], [862, 760], [861, 803], [732, 799]], 'text': '76,000', 'confidence': 0.9345720410346985, 'matches_source': None, 'review_note': ''}, {'box': [[241, 837], [587, 841], [586, 880], [241, 875]], 'text': '부기세 과세국기액', 'confidence': 0.6522305011749268, 'matches_source': None, 'review_note': ''}, {'box': [[733, 838], [854, 843], [853, 886], [731, 881]], 'text': '639,094', 'confidence': 0.8303846716880798, 'matches_source': None, 'review_note': ''}, {'box': [[240, 881], [278, 881], [278, 919], [240, 919]], 'text': '', 'confidence': 0.0, 'matches_source': None, 'review_note': ''}, {'box': [[406, 883], [444, 883], [444, 918], [406, 918]], 'text': '|', 'confidence': 0.4424668550491333, 'matches_source': None, 'review_note': ''}, {'box': [[551, 882], [590, 882], [590, 920], [551, 920]], 'text': '시', 'confidence': 0.977571427822113, 'matches_source': None, 'review_note': ''}, {'box': [[748, 880], [854, 884], [853, 929], [747, 925]], 'text': '6,906', 'confidence': 0.9977938532829285, 'matches_source': None, 'review_note': ''}, {'box': [[161, 962], [427, 967], [426, 1002], [161, 998]], 'text': '** 현영수', 'confidence': 0.8044949173927307, 'matches_source': None, 'review_note': ''}, {'box': [[452, 968], [667, 968], [667, 999], [452, 999]], 'text': '3세)[1]', 'confidence': 0.6715445518493652, 'matches_source': None, 'review_note': ''}, {'box': [[674, 971], [736, 971], [736, 997], [674, 997]], 'text': '**', 'confidence': 0.6336501836776733, 'matches_source': None, 'review_note': ''}]
recorded_text = "\n".join(
    reconstruct_spatial_lines(RECORDED_PP_OCRV5_TOKENS)
)
recorded_receipt = extract_receipt_from_text(
    recorded_text,
    "recorded_ppocrv5_regression",
)
assert recorded_receipt["date"] == "2025-10-04"
assert recorded_receipt["total_amount"] == 76000
assert len(recorded_receipt["items"]) == 5
print(
    "RECORDED LIVE REGRESSION PASS:",
    recorded_receipt["total_amount"],
    len(recorded_receipt["items"]),
)


## 내가 직접 정하는 LIVE 통과 조건 3개

앱이 오류 없이 열리는 것과 추출값이 맞는 것은 다릅니다. LIVE 경로가
반드시 확인해야 할 값을 세 개 고릅니다.


In [ ]:
# TODO: 확인할 필드 세 개를 채우세요.
my_live_checks = [None, None, None]
if any(value is None for value in my_live_checks):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")


<details>
<summary>힌트와 전체 정답 보기</summary>

날짜, 총액, 반복 품목 수는 후속 검증과 Excel에 직접 영향을 줍니다.
</details>


In [ ]:
ANSWER_LIVE_CHECKS = ["date", "total_amount", "items"]
assert recorded_receipt["date"]
assert recorded_receipt["total_amount"] is not None
assert recorded_receipt["items"]
print("전체 정답 · LIVE 필수 확인:", ANSWER_LIVE_CHECKS)


In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 연결 앱"
assert len(app_test.button) == 2
app_test.button[1].click().run(timeout=20)
assert any("PREPARED_FALLBACK" in item.value for item in app_test.success)
assert app_test.json
print("CHECKPOINT 1/1 PASS: 앱 연결·모드 표시·JSON 출력")


In [ ]:
# 선택 실습 · 녹화에서는 이 셀로 실제 화면을 엽니다.
# AppTest가 필수 검증이며, 미리보기에는 공개 비식별 샘플만 사용합니다.
if not VALIDATION_MODE:
    import subprocess
    import time
    import urllib.request

    preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(output_path),
            "--server.port", "8506",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            urllib.request.urlopen(
                "http://127.0.0.1:8506/_stcore/health",
                timeout=1,
            )
            break
        except Exception:
            time.sleep(0.5)
    try:
        from google.colab import output
        print("아래 화면에서 직접 버튼과 입력값을 조작하세요.")
        output.serve_kernel_port_as_iframe(8506, height=760)
    except Exception as exc:
        print("Colab 미리보기를 열지 못했습니다:", exc)
        print("AppTest 결과와 app 파일로 계속합니다.")
else:
    print("검증 모드: 대화형 Streamlit 미리보기 생략")
